In [1]:
import pandas as pd

transactions = pd.read_csv('../data/raw/train_transaction.csv')
transactions.shape

(590540, 394)

In [2]:
transactions.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
transactions['isFraud'].value_counts(normalize=True)

isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64

In [4]:
transactions.isnull().sum().sort_values(ascending=False).head(20)

dist2    552913
D7       551623
D13      528588
D14      528353
D12      525823
D6       517353
D9       515614
D8       515614
V153     508595
V139     508595
V162     508595
V161     508595
V154     508595
V138     508595
V158     508595
V157     508595
V163     508595
V156     508595
V155     508595
V149     508595
dtype: int64

In [5]:
import pandas as pd
identity = pd.read_csv('../data/raw/train_identity.csv')
identity.shape


(144233, 41)

In [6]:
df = transactions.merge(identity, on='TransactionID', how='left')
df.shape

(590540, 434)

In [7]:
missing_pct= (df.isnull().sum()/len(df))*100
missing_pct.sort_values(ascending=False).head(30)

id_24    99.196159
id_25    99.130965
id_07    99.127070
id_08    99.127070
id_21    99.126393
id_26    99.125715
id_27    99.124699
id_23    99.124699
id_22    99.124699
dist2    93.628374
D7       93.409930
id_18    92.360721
D13      89.509263
D14      89.469469
D12      89.041047
id_03    88.768923
id_04    88.768923
D6       87.606767
id_33    87.589494
id_10    87.312290
id_09    87.312290
D9       87.312290
D8       87.312290
id_30    86.865411
id_32    86.861855
id_34    86.824771
id_14    86.445626
V142     86.123717
V158     86.123717
V140     86.123717
dtype: float64

In [8]:
cols_to_drop=missing_pct[missing_pct>90].index.tolist()
len(cols_to_drop)

12

In [9]:
cols_to_drop_85=missing_pct[missing_pct>85].index.tolist()
len(cols_to_drop_85)

74

In [10]:
df_clean=df.drop(columns=cols_to_drop_85)
df_clean.shape

(590540, 360)

In [12]:
missing_pct_clean=(df_clean.isnull().sum()/len(df_clean))*100
missing_pct_clean.sort_values(ascending=False).head(15)

DeviceInfo    79.905510
id_13         78.440072
id_16         78.098012
V276          77.913435
V262          77.913435
V249          77.913435
V252          77.913435
V253          77.913435
V254          77.913435
V257          77.913435
V258          77.913435
V260          77.913435
V261          77.913435
V263          77.913435
V275          77.913435
dtype: float64

In [13]:
df_clean.shape

(590540, 360)

In [15]:
numeric_cols=df_clean.select_dtypes(include=['float64','int64']).columns.tolist()
categorical_cols=df_clean.select_dtypes(include=['object','str']).columns.tolist()
print(len(numeric_cols), len(categorical_cols))

334 26


In [16]:
df_clean[numeric_cols]=df_clean[numeric_cols].fillna(-999)

In [17]:
df_clean[categorical_cols]=df_clean[categorical_cols].fillna('missing')

In [18]:
df_clean.isnull().sum().sum()

np.int64(0)

In [19]:
for col in categorical_cols:
    print(col,df_clean[col].nunique())

ProductCD 5
card4 5
card6 5
P_emaildomain 60
R_emaildomain 61
M1 3
M2 3
M3 3
M4 4
M5 3
M6 3
M7 3
M8 3
M9 3
id_12 3
id_15 4
id_16 3
id_28 3
id_29 3
id_31 131
id_35 3
id_36 3
id_37 3
id_38 3
DeviceType 3
DeviceInfo 1787


In [23]:
device_counts = df_clean['DeviceInfo'].value_counts()
rare_devices = device_counts[device_counts<100].index
df_clean['DeviceInfo'] = df_clean['DeviceInfo'].replace(rare_devices,'other')
df_clean['DeviceInfo'].nunique()

65

In [26]:
from sklearn.preprocessing import LabelEncoder

label_encoders={}
for col in categorical_cols:
    le=LabelEncoder()
    df_clean[col]=le.fit_transform(df_clean[col])
    label_encoders[col]=le

df_clean.shape

(590540, 360)

In [27]:
df_clean.dtypes.value_counts()

float64    330
int64       30
Name: count, dtype: int64

In [ ]:
df_clean.to_csv('../data/processed/train_processed.csv', index=False)